# PR. 9 Visualizer — Pandas Analyzer & Data Visualization

## 1. Imports

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline


## 2. The `SalesDataAnalyzer` Class

In [5]:
class SalesDataAnalyzer:
    """
    Encapsulates loading, cleaning, analyzing and visualizing a sales dataset.
    """

    # ---------------------------------------------------------------- #
    # Construction / Destruction
    # ---------------------------------------------------------------- #
    def __init__(self, file_path=None):
        self.data = None            # pandas DataFrame holding the sales data
        self.current_fig = None     # last generated Matplotlib figure
        if file_path:
            self.load_data(file_path)

    def __del__(self):
        # Cleanup: close any open Matplotlib figures
        try:
            plt.close("all")
        except Exception:
            pass

    # ---------------------------------------------------------------- #
    # Data Acquisition
    # ---------------------------------------------------------------- #
    def load_data(self, file_path):
        """Load data from a CSV file. Offers to generate a sample dataset if missing."""
        if not os.path.exists(file_path):
            print(f"\nFile not found at: {file_path}")
            choice = input("Generate a sample dataset at this location? (y/n): ").strip().lower()
            if choice == "y":
                self._generate_sample_data(file_path)
            else:
                print("Load cancelled.")
                return
        try:
            self.data = pd.read_csv(file_path, parse_dates=["Date"], dayfirst=False)
        except (ValueError, KeyError):
            # Date column might not exist / not parseable - load without it
            self.data = pd.read_csv(file_path)
        except FileNotFoundError:
            print("Error: file not found.")
            return
        except Exception as e:
            print(f"Error loading dataset: {e}")
            return
        print("Dataset loaded successfully!")

    @staticmethod
    def _generate_sample_data(file_path, n=100):
        """Create a small synthetic sales dataset and save it as CSV."""
        np.random.seed(42)
        products = ["Product A", "Product B", "Product C", "Product D", "Product E"]
        regions = ["North", "South", "East", "West", "Central"]
        dates = pd.date_range("2022-01-01", periods=n, freq="D")

        df = pd.DataFrame({
            "SalesID": range(101, 101 + n),
            "Date": dates,
            "Product": np.random.choice(products, n),
            "Region": np.random.choice(regions, n),
            "Sales": np.random.randint(100, 1000, n),
            "Profit": np.random.randint(10, 300, n),
        })
        df["Year"] = df["Date"].dt.year

        folder = os.path.dirname(file_path)
        if folder:
            os.makedirs(folder, exist_ok=True)
        df.to_csv(file_path, index=False)
        print(f"Sample dataset generated at {file_path}")

    # ---------------------------------------------------------------- #
    # Data Exploration
    # ---------------------------------------------------------------- #
    def explore_data(self):
        if not self._has_data():
            return
        print("\n== Explore Data ==")
        print("1. Display the first 5 rows")
        print("2. Display the last 5 rows")
        print("3. Display column names")
        print("4. Display data types")
        print("5. Basic info")
        choice = input("Enter your choice: ").strip()

        if choice == "1":
            print(self.data.head())
        elif choice == "2":
            print(self.data.tail())
        elif choice == "3":
            print(list(self.data.columns))
        elif choice == "4":
            print(self.data.dtypes)
        elif choice == "5":
            self.data.info()
            print(self.data.describe())
        else:
            print("Invalid choice.")

    # ---------------------------------------------------------------- #
    # Data Cleaning
    # ---------------------------------------------------------------- #
    def handle_missing_data(self):
        if not self._has_data():
            return
        print("\n== Handle Missing Data ==")
        print("1. Display rows with missing values")
        print("2. Fill missing values with mean")
        print("3. Drop rows with missing values")
        print("4. Replace missing values with a specific value")
        choice = input("Enter your choice: ").strip()

        if self.data.isnull().sum().sum() == 0 and choice in ("2", "3", "4"):
            print("\nNo missing values found in the dataset!")
            return

        if choice == "1":
            missing = self.data[self.data.isnull().any(axis=1)]
            if missing.empty:
                print("\nNo missing values found in the dataset!")
            else:
                print(missing)
        elif choice == "2":
            numeric_cols = self.data.select_dtypes(include=np.number).columns
            self.data[numeric_cols] = self.data[numeric_cols].fillna(
                self.data[numeric_cols].mean()
            )
            print("Missing numeric values filled with column mean.")
        elif choice == "3":
            before = len(self.data)
            self.data = self.data.dropna()
            print(f"Dropped {before - len(self.data)} rows containing missing values.")
        elif choice == "4":
            value = input("Enter the value to replace missing entries with: ")
            self.data = self.data.fillna(value)
            print(f"Missing values replaced with '{value}'.")
        else:
            print("Invalid choice.")

    # ---------------------------------------------------------------- #
    # DataFrame / NumPy Operations
    # ---------------------------------------------------------------- #
    def dataframe_operations_menu(self):
        if not self._has_data():
            return
        print("\n== Perform DataFrame Operations ==")
        print("1. NumPy Array: Creation, Indexing & Slicing")
        print("2. Mathematical Operations (element-wise)")
        print("3. Combine Data (concat with a copy of itself)")
        print("4. Split Data (by Region)")
        print("5. Search / Sort / Filter")
        print("6. Aggregate Functions (sum, mean, count)")
        print("7. Create Pivot Table")
        choice = input("Enter your choice: ").strip()

        if choice == "1":
            self._numpy_array_ops()
        elif choice == "2":
            self._mathematical_operations()
        elif choice == "3":
            self.combine_data(self.data.copy())
        elif choice == "4":
            self.split_data()
        elif choice == "5":
            self.search_sort_filter()
        elif choice == "6":
            self.aggregate_functions()
        elif choice == "7":
            self.create_pivot_table()
        else:
            print("Invalid choice.")

    def _numpy_array_ops(self):
        arr = self.data["Sales"].to_numpy()
        print(f"\nNumPy array of 'Sales' (first 10 shown): {arr[:10]}")
        print(f"Shape: {arr.shape}, dtype: {arr.dtype}")
        print(f"Indexing -> arr[0] = {arr[0]}")
        print(f"Slicing  -> arr[2:6] = {arr[2:6]}")
        print(f"Negative indexing -> arr[-1] = {arr[-1]}")

    def _mathematical_operations(self):
        sales = self.data["Sales"].to_numpy()
        profit = self.data["Profit"].to_numpy() if "Profit" in self.data.columns else None
        print(f"\nSales + 10 (first 5): {(sales + 10)[:5]}")
        print(f"Sales * 1.1 [10% increase] (first 5): {(sales * 1.1)[:5]}")
        if profit is not None:
            margin = np.divide(profit, sales, out=np.zeros_like(profit, dtype=float), where=sales != 0)
            print(f"Profit margin (Profit / Sales), first 5: {np.round(margin[:5], 3)}")

    def combine_data(self, other_dataframe):
        combined = pd.concat([self.data, other_dataframe], ignore_index=True)
        print(f"\nCombined dataset shape: {combined.shape} (original: {self.data.shape})")
        keep = input("Keep combined dataset as current data? (y/n): ").strip().lower()
        if keep == "y":
            self.data = combined
            print("Data combined and updated.")

    def split_data(self):
        if "Region" not in self.data.columns:
            print("No 'Region' column found to split on.")
            return
        groups = {region: df for region, df in self.data.groupby("Region")}
        print(f"\nSplit into {len(groups)} DataFrames by Region:")
        for region, df in groups.items():
            print(f"  {region}: {len(df)} rows")

    def search_sort_filter(self):
        print("\n1. Search by Product name")
        print("2. Sort by Sales (descending)")
        print("3. Filter by Region")
        choice = input("Enter your choice: ").strip()
        if choice == "1":
            product = input("Enter product name to search: ").strip()
            result = self.data[self.data["Product"].str.contains(product, case=False, na=False)]
            print(result if not result.empty else "No matching records found.")
        elif choice == "2":
            print(self.data.sort_values(by="Sales", ascending=False).head(10))
        elif choice == "3":
            region = input("Enter region to filter by: ").strip()
            result = self.data[self.data["Region"].str.contains(region, case=False, na=False)]
            print(result if not result.empty else "No matching records found.")
        else:
            print("Invalid choice.")

    def aggregate_functions(self):
        numeric_cols = self.data.select_dtypes(include=np.number).columns
        print("\nSum:\n", self.data[numeric_cols].sum())
        print("\nMean:\n", self.data[numeric_cols].mean())
        print("\nCount:\n", self.data[numeric_cols].count())

    def create_pivot_table(self):
        if not {"Region", "Product", "Sales"}.issubset(self.data.columns):
            print("Required columns (Region, Product, Sales) not found.")
            return
        pivot = pd.pivot_table(self.data, values="Sales", index="Region",
                                columns="Product", aggfunc="sum", fill_value=0)
        print("\nPivot Table (Sum of Sales by Region & Product):")
        print(pivot)

    # ---------------------------------------------------------------- #
    # Statistical Analysis
    # ---------------------------------------------------------------- #
    def statistical_analysis(self):
        if not self._has_data():
            return
        numeric_cols = self.data.select_dtypes(include=np.number).columns
        print("\n== Descriptive Statistics ==")
        print(self.data[numeric_cols].describe())
        print("\nStandard Deviation:\n", self.data[numeric_cols].std())
        print("\nVariance:\n", self.data[numeric_cols].var())
        print("\n25th/50th/75th Percentiles:\n", self.data[numeric_cols].quantile([0.25, 0.5, 0.75]))

    # ---------------------------------------------------------------- #
    # Data Visualization (Matplotlib + Seaborn)
    # ---------------------------------------------------------------- #
    def visualize_data(self):
        if not self._has_data():
            return
        print("\n== Data Visualization ==")
        print("1. Bar Plot")
        print("2. Line Plot")
        print("3. Scatter Plot")
        print("4. Pie Chart")
        print("5. Histogram")
        print("6. Stack Plot")
        print("7. Heatmap (Seaborn)")
        print("8. Box Plot (Seaborn)")
        choice = input("Enter your choice: ").strip()

        plt.close("all")
        fig, ax = plt.subplots(figsize=(8, 5))

        if choice == "1":
            print("\n== Bar Plot ==")
            self.data.groupby("Product")["Sales"].sum().plot(kind="bar", ax=ax, color="steelblue")
            ax.set_title("Total Sales by Product")
            ax.set_ylabel("Sales")

        elif choice == "2":
            print("\n== Line Plot ==")
            x_col = input("Enter x-axis column name: ").strip()
            y_col = input("Enter y-axis column name: ").strip()
            print("Generating line plot...")
            ax.plot(self.data[x_col], self.data[y_col], marker="o")
            ax.set_title(f"{y_col} over {x_col}")
            ax.set_xlabel(x_col)
            ax.set_ylabel(y_col)
            print("Line plot displayed successfully!")

        elif choice == "3":
            print("\n== Scatter Plot ==")
            x_col = input("Enter x-axis column name: ").strip()
            y_col = input("Enter y-axis column name: ").strip()
            print("Generating scatter plot...")
            ax.scatter(self.data[x_col], self.data[y_col], alpha=0.7, color="darkorange")
            ax.set_title(f"{y_col} vs {x_col}")
            ax.set_xlabel(x_col)
            ax.set_ylabel(y_col)
            print("Scatter plot displayed successfully!")

        elif choice == "4":
            print("\n== Pie Chart ==")
            sums = self.data.groupby("Region")["Sales"].sum()
            ax.pie(sums, labels=sums.index, autopct="%1.1f%%")
            ax.set_title("Sales Share by Region")

        elif choice == "5":
            print("\n== Histogram ==")
            col = input("Enter column name for histogram: ").strip()
            ax.hist(self.data[col], bins=10, color="mediumseagreen", edgecolor="black")
            ax.set_title(f"Distribution of {col}")
            ax.set_xlabel(col)
            ax.set_ylabel("Frequency")

        elif choice == "6":
            print("\n== Stack Plot ==")
            pivot = self.data.pivot_table(values="Sales", index="Year", columns="Region",
                                           aggfunc="sum", fill_value=0)
            ax.stackplot(pivot.index, pivot.T.values, labels=pivot.columns)
            ax.set_title("Sales by Region over Time (Stacked)")
            ax.legend(loc="upper left")

        elif choice == "7":
            print("\n== Heatmap (Seaborn) ==")
            numeric_cols = self.data.select_dtypes(include=np.number).columns
            sns.heatmap(self.data[numeric_cols].corr(), annot=True, cmap="coolwarm", ax=ax)
            ax.set_title("Correlation Heatmap")

        elif choice == "8":
            print("\n== Box Plot (Seaborn) ==")
            sns.boxplot(data=self.data, x="Region", y="Sales", ax=ax)
            ax.set_title("Sales Distribution by Region")

        else:
            print("Invalid choice.")
            plt.close(fig)
            return

        fig.tight_layout()
        self.current_fig = fig
        plt.show()

    # ---------------------------------------------------------------- #
    # Save Visualization
    # ---------------------------------------------------------------- #
    def save_visualization(self):
        print("\n== Save Visualization ==")
        if self.current_fig is None:
            print("No visualization to save. Please generate one first.")
            return
        filename = input("Enter the file name to save the plot (e.g., scatter_plot.png): ").strip()
        try:
            self.current_fig.savefig(filename)
            print(f"Visualization saved as {filename} successfully!")
        except Exception as e:
            print(f"Error saving visualization: {e}")

    # ---------------------------------------------------------------- #
    # Helpers
    # ---------------------------------------------------------------- #
    def _has_data(self):
        if self.data is None:
            print("\nNo dataset loaded. Please load a dataset first (Option 1).")
            return False
        return True


## 3. Program Flow — Main Menu

In [7]:
def print_main_menu():
    print("\n========== Data Analysis & Visualization Program ==========")
    print("Please select an option:")
    print("1. Load Dataset")
    print("2. Explore Data")
    print("3. Perform DataFrame Operations")
    print("4. Handle Missing Data")
    print("5. Generate Descriptive Statistics")
    print("6. Data Visualization")
    print("7. Save Visualization")
    print("8. Exit")
    print("=" * 60)


def main():
    analyzer = SalesDataAnalyzer()

    while True:
        print_main_menu()
        choice = input("\nEnter your choice: ").strip()

        if choice == "1":
            path = input("Enter the path of the dataset (CSV file): ").strip()
            analyzer.load_data(path)
        elif choice == "2":
            analyzer.explore_data()
        elif choice == "3":
            analyzer.dataframe_operations_menu()
        elif choice == "4":
            analyzer.handle_missing_data()
        elif choice == "5":
            analyzer.statistical_analysis()
        elif choice == "6":
            analyzer.visualize_data()
        elif choice == "7":
            analyzer.save_visualization()
        elif choice == "8":
            print("\nExiting the program. Goodbye!")
            break
        else:
            print("Invalid choice, please try again.")

    return analyzer


## 4. Run the Program

Running the cell below starts the interactive menu (uses `input()`, so it works in a Jupyter cell just like a console). Enter option numbers as prompted, e.g. `1` to load a dataset (try `data/sales_data.csv`, or let it generate a sample for you).

In [ ]:
analyzer = main()



========== Data Analysis & Visualization Program ==========
Please select an option:
1. Load Dataset
2. Explore Data
3. Perform DataFrame Operations
4. Handle Missing Data
5. Generate Descriptive Statistics
6. Data Visualization
7. Save Visualization
8. Exit



Enter your choice:  1
